# Varroa Mite Detection - Modular Notebook

This notebook is generated from the original all-in-one experiment notebook.

Outputs are intentionally cleared. API keys and personal secrets are not stored in this notebook.

## YOLO Object Detection Experiments

Models: YOLOv8n and YOLO26n

Tasks:
- Dataset download
- Single-class YOLO preprocessing
- Internal validation
- External validation
- Result graphs


In [ ]:
!pip install -q ultralytics roboflow pyyaml pandas matplotlib opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
import shutil
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO

PROJECT_DIR = '/content/drive/MyDrive/varroa_bitirme'
RAW_DIR = f'{PROJECT_DIR}/raw_datasets'
PROCESSED_DIR = f'{PROJECT_DIR}/processed_datasets'
RUNS_DIR = f'{PROJECT_DIR}/runs'
GRAPH_DIR = f'{PROJECT_DIR}/github_cnn_yolo_outputs'

for d in [RAW_DIR, PROCESSED_DIR, RUNS_DIR, GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('RUNS_DIR:', RUNS_DIR)
print('GRAPH_DIR:', GRAPH_DIR)

# Do not hard-code API keys in GitHub.
ROBOFLOW_API_KEY = getpass('Roboflow API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
def download_datasets():
    os.chdir(RAW_DIR)

    project_varroa = rf.workspace('varroa-j8231').project('varroa-bxxhd')
    dataset_varroa = project_varroa.version(1).download('yolov8')

    project_honeybee = rf.workspace('honeybee').project('honeybee_varroamite')
    dataset_honeybee = project_honeybee.version(5).download('yolov8')

    project_external = rf.workspace('varroa-virus-detection').project('varroa-mites-detector')
    dataset_external = project_external.version(2).download('yolov8')

    return dataset_varroa.location, dataset_honeybee.location, dataset_external.location

VARROA_RAW, HONEYBEE_RAW, EXTERNAL_RAW = download_datasets()

print('VARROA_RAW:', VARROA_RAW)
print('HONEYBEE_RAW:', HONEYBEE_RAW)
print('EXTERNAL_RAW:', EXTERNAL_RAW)

In [ ]:
def prepare_single_class_yolo_dataset(src_dir, dst_dir, source_varroa_class_ids, new_class_name='varroa', min_wh=1e-6, max_area=0.60):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)

    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    stats = {
        'copied_images': 0,
        'total_label_lines': 0,
        'kept_label_lines': 0,
        'removed_wrong_class': 0,
        'removed_invalid_bbox': 0,
        'empty_labels_after_cleaning': 0,
    }

    for split in ['train', 'valid', 'test']:
        src_img_dir = src_dir / split / 'images'
        src_lbl_dir = src_dir / split / 'labels'
        dst_img_dir = dst_dir / split / 'images'
        dst_lbl_dir = dst_dir / split / 'labels'
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        if not src_img_dir.exists():
            continue

        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
            image_paths.extend(src_img_dir.glob(ext))

        for img_path in image_paths:
            shutil.copy2(img_path, dst_img_dir / img_path.name)
            stats['copied_images'] += 1

            label_path = src_lbl_dir / f'{img_path.stem}.txt'
            new_lines = []

            if label_path.exists():
                lines = label_path.read_text().strip().splitlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        stats['removed_invalid_bbox'] += 1
                        continue

                    stats['total_label_lines'] += 1
                    class_id = int(float(parts[0]))
                    x, y, w, h = map(float, parts[1:])

                    if class_id not in source_varroa_class_ids:
                        stats['removed_wrong_class'] += 1
                        continue

                    if w <= min_wh or h <= min_wh or (w * h) > max_area:
                        stats['removed_invalid_bbox'] += 1
                        continue

                    new_lines.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
                    stats['kept_label_lines'] += 1

            if len(new_lines) == 0:
                stats['empty_labels_after_cleaning'] += 1

            (dst_lbl_dir / f'{img_path.stem}.txt').write_text('\n'.join(new_lines))

    data_yaml = {
        'path': str(dst_dir),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': {0: new_class_name}
    }
    with open(dst_dir / 'data.yaml', 'w') as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    return stats

In [ ]:
VARROA_PROCESSED = f'{PROCESSED_DIR}/varroa_dataset_single_class'
HONEYBEE_PROCESSED = f'{PROCESSED_DIR}/honeybee_varroamite_single_class'
EXTERNAL_PROCESSED = f'{PROCESSED_DIR}/external_varroa_mites_detector_single_class'

stats_varroa = prepare_single_class_yolo_dataset(VARROA_RAW, VARROA_PROCESSED, source_varroa_class_ids=[1], max_area=0.35)
stats_honeybee = prepare_single_class_yolo_dataset(HONEYBEE_RAW, HONEYBEE_PROCESSED, source_varroa_class_ids=[0], max_area=0.60)
stats_external = prepare_single_class_yolo_dataset(EXTERNAL_RAW, EXTERNAL_PROCESSED, source_varroa_class_ids=[0], max_area=0.35)

print('Varroa stats:', stats_varroa)
print('HoneyBee stats:', stats_honeybee)
print('External stats:', stats_external)

In [ ]:
def train_yolo(model_name, data_yaml, run_name, epochs=30, imgsz=640, batch=16):
    model = YOLO(model_name)
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project=RUNS_DIR,
        name=run_name,
        device=0,
        workers=1,
        patience=5
    )
    return results

def validate_yolo(weights_path, data_yaml, run_name, imgsz=640, batch=16):
    model = YOLO(weights_path)
    results = model.val(
        data=data_yaml,
        imgsz=imgsz,
        batch=batch,
        project=RUNS_DIR,
        name=run_name,
        device=0,
        workers=1
    )
    return results

# YOLOv8n - Varroa Dataset
train_yolo(
    model_name='yolov8n.pt',
    data_yaml=f'{VARROA_PROCESSED}/data.yaml',
    run_name='yolov8n_varroa_dataset_pilot',
    epochs=30
)

yolov8n_varroa_external = validate_yolo(
    weights_path=f'{RUNS_DIR}/yolov8n_varroa_dataset_pilot/weights/best.pt',
    data_yaml=f'{EXTERNAL_PROCESSED}/data.yaml',
    run_name='external_val_yolov8n_varroa_dataset_pilot'
)

# YOLO26n - Varroa Dataset
train_yolo(
    model_name='yolo26n.pt',
    data_yaml=f'{VARROA_PROCESSED}/data.yaml',
    run_name='yolo26n_varroa_dataset_pilot',
    epochs=30
)

yolo26n_varroa_external = validate_yolo(
    weights_path=f'{RUNS_DIR}/yolo26n_varroa_dataset_pilot/weights/best.pt',
    data_yaml=f'{EXTERNAL_PROCESSED}/data.yaml',
    run_name='external_val_yolo26n_varroa_dataset_pilot'
)

# YOLOv8n - HoneyBee VarroaMite
train_yolo(
    model_name='yolov8n.pt',
    data_yaml=f'{HONEYBEE_PROCESSED}/data.yaml',
    run_name='yolov8n_honeybee_varroamite_pilot',
    epochs=30
)

yolov8n_honeybee_external = validate_yolo(
    weights_path=f'{RUNS_DIR}/yolov8n_honeybee_varroamite_pilot/weights/best.pt',
    data_yaml=f'{EXTERNAL_PROCESSED}/data.yaml',
    run_name='external_val_yolov8n_honeybee_varroamite_pilot'
)

# YOLO26n - HoneyBee VarroaMite
train_yolo(
    model_name='yolo26n.pt',
    data_yaml=f'{HONEYBEE_PROCESSED}/data.yaml',
    run_name='yolo26n_honeybee_varroamite_pilot',
    epochs=30
)

yolo26n_honeybee_external = validate_yolo(
    weights_path=f'{RUNS_DIR}/yolo26n_honeybee_varroamite_pilot/weights/best.pt',
    data_yaml=f'{EXTERNAL_PROCESSED}/data.yaml',
    run_name='external_val_yolo26n_honeybee_varroamite_pilot'
)

# Manual summary table based on recorded experiment outputs.
yolo_data = [
    {'Model':'YOLOv8n','Dataset':'Varroa','Evaluation':'Internal','Precision':0.890,'Recall':0.775,'F1-score':0.829,'mAP50':0.860,'mAP50-95':0.368},
    {'Model':'YOLOv8n','Dataset':'Varroa','Evaluation':'External','Precision':0.817,'Recall':0.549,'F1-score':0.657,'mAP50':0.629,'mAP50-95':0.222},
    {'Model':'YOLOv8n','Dataset':'HoneyBee','Evaluation':'Internal','Precision':0.898,'Recall':0.695,'F1-score':0.783,'mAP50':0.750,'mAP50-95':0.291},
    {'Model':'YOLOv8n','Dataset':'HoneyBee','Evaluation':'External','Precision':0.133,'Recall':0.303,'F1-score':0.185,'mAP50':0.0885,'mAP50-95':0.0195},
    {'Model':'YOLO26n','Dataset':'Varroa','Evaluation':'Internal','Precision':0.758,'Recall':0.695,'F1-score':0.725,'mAP50':0.787,'mAP50-95':0.344},
    {'Model':'YOLO26n','Dataset':'Varroa','Evaluation':'External','Precision':0.625,'Recall':0.496,'F1-score':0.553,'mAP50':0.557,'mAP50-95':0.208},
    {'Model':'YOLO26n','Dataset':'HoneyBee','Evaluation':'Internal','Precision':0.850,'Recall':0.681,'F1-score':0.756,'mAP50':0.748,'mAP50-95':0.275},
    {'Model':'YOLO26n','Dataset':'HoneyBee','Evaluation':'External','Precision':0.189,'Recall':0.268,'F1-score':0.222,'mAP50':0.093,'mAP50-95':0.0217},
]

df_yolo = pd.DataFrame(yolo_data)
df_yolo['Model_Dataset'] = df_yolo['Model'] + ' - ' + df_yolo['Dataset']
df_yolo.to_csv(f'{GRAPH_DIR}/yolo_results_summary.csv', index=False)
df_yolo

In [ ]:
def plot_metric(df, metric, title, filename):
    plot_df = df.pivot_table(index='Model_Dataset', columns='Evaluation', values=metric, aggfunc='first')
    ax = plot_df[['Internal', 'External']].plot(kind='bar', figsize=(11, 5))
    plt.title(title)
    plt.xlabel('Model and training dataset')
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.legend(title='Evaluation')
    plt.tight_layout()
    save_path = f'{GRAPH_DIR}/{filename}'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', save_path)

plot_metric(df_yolo, 'Recall', 'YOLO Models Internal and External Recall Comparison', 'yolo_recall_comparison.png')
plot_metric(df_yolo, 'F1-score', 'YOLO Models Internal and External F1-score Comparison', 'yolo_f1_comparison.png')
plot_metric(df_yolo, 'mAP50', 'YOLO Models Internal and External mAP50 Comparison', 'yolo_map50_comparison.png')
plot_metric(df_yolo, 'mAP50-95', 'YOLO Models Internal and External mAP50-95 Comparison', 'yolo_map50_95_comparison.png')